## Data Exploration and Cleaning
\
In this phase I am loading in the near earth asteroid dataset which can be found [here](https://www.kaggle.com/datasets/darkmatternet/nasa-near-earth-asteroids-and-close-approaches?select=near_earth_asteroids_2025+%281%29.csv) and performed initial profiling to understand its structure, distributions, and cleaning requirements. Specifically, I tackled the following:
* Loading and displaying rows, counts, types, etc.
* Looked at the frequecy of different classes (I knew I was going to be interested in the orbital classes initially).
* Cleaned the data. I wanted to use the first observed (aka discovery date) so I check that for null values and surprisingly it came back with 1 so I dropped that


### Notes
* I ran into an initial warning when loading the data which can be resolved by using the low_memory=False in panadas read csv
* The different orbital classes do have meaning and will be discussed later in the document
* The sizes are quite long in their description so I decided to shorten those for the visuals
* I also added a new column for just the standalone years because I knew in my visual I was not going to care about the specific date.

In [12]:
import pandas as pd
import plotly.express as px

# Passing low_memory=False will get rid of the warning
df = pd.read_csv("/voc/work/near_earth_asteroids_2025.csv", low_memory=False)

print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")
print("-" * 50)

df.head(3)
df.info()

Total Rows: 41,281
Total Columns: 29
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41281 entries, 0 to 41280
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   spkid                  41281 non-null  int64  
 1   full_name              41281 non-null  object 
 2   pdes                   41281 non-null  object 
 3   name                   182 non-null    object 
 4   pha                    41281 non-null  bool   
 5   H                      41279 non-null  float64
 6   diameter_km            41281 non-null  float64
 7   diameter_m             41281 non-null  float64
 8   diameter_is_estimated  41281 non-null  bool   
 9   size_category          41281 non-null  object 
 10  albedo                 1204 non-null   float64
 11  rot_per                2181 non-null   float64
 12  class                  41281 non-null  object 
 13  e                 

In [13]:
## Checking out the distribution of different orbital classes
# APO (Apollo)
# AMO (Armor)
# ATE (Aten)
# IEO (Interior to Earth)

orbit_col = 'class'

print(df[orbit_col].value_counts())

class
APO    23484
AMO    14408
ATE     3351
IEO       38
Name: count, dtype: int64


In [14]:
## Checking out the distribution over size

size = 'size_category'

print(df[size].value_counts())

size_category
Small (25-140m) — Local damage         19154
Medium (140m-1km) — Regional damage    10726
Tiny (<25m) — Airburst/harmless        10460
Large (>1 km) — City killer+             941
Name: count, dtype: int64


In [15]:
missing_dates = df['first_obs'].isna().sum()

print(missing_dates)

## Check out how the first oberved dates are formatted

print(df['first_obs'].head(10))

1
0    1893-10-29
1    1911-10-04
2    1918-02-09
3    1924-10-23
4    1932-03-12
5    1949-07-01
6    1952-10-27
7    1951-09-14
8    1929-09-25
9    1948-07-17
Name: first_obs, dtype: object


In [16]:
df_clean = df.dropna(subset=['first_obs']).copy()

df_clean['discovery_year'] = df_clean['first_obs'].str[:4].astype(int)

size_mapping = {
    "Tiny (<25m) — Airburst/harmless": "Airburst/harmless",
    "Small (25-140m) — Local damage": "Local Damage",
    "Medium (140m-1km) — Regional damage": "Regional Damage",
    "Large (>1 km) — City killer+": "City Killer+"
}

df_clean['size_short'] = df_clean['size_category'].map(size_mapping)

df_clean[['first_obs', 'discovery_year', 'size_short']].head()

,first_obs,discovery_year,size_short
0,1893-10-29,1893,City Killer+
1,1911-10-04,1911,City Killer+
2,1918-02-09,1918,City Killer+
3,1924-10-23,1924,City Killer+
4,1932-03-12,1932,City Killer+


## Visualization Techniques

### Why I chose the data:
I got my degree in physics for my undergrad and so naturally space has always interested me. In my undergrad we breifly spoke about NEOs and how typically when people hear that term they either freak out or do not know what it means. So I chose this dataset to work with since it offers some fairly clean data from what I can tell and it is cool to see that yes there are some threats but also they are a very very small number compared to the rest. No armageddan today.

### Figure 1
I wanted to look at the relationship between the some of the orbital properties of the asteroids (orbital distance and orbital shape). To do this I looked at the following:
* x-axis: Semi-Major Axis (a) which tells up the distance of the orbit from the sun
* y-axis: Eccentricity (e) which tells us the orbital shape (e = 0 means circle and e = 1 is a very stretched ellipse
* The coloring I chose was something NASA calls Potentially Hazardous Asteroid (pha) with True meaning we should probably keep an eye on it and False meaning it is low risk.


In [17]:
fig1 = px.scatter(
    df_clean,
    x='a',
    y='e',
    color='pha',
    title = 'Orbits: Semi-Major Axis vs Eccentricity',
    labels={'a': 'Semi-Major Axis (AU)', 'e': 'Eccentricity', 'pha': 'Potentially Hazardous'}
)

fig1.update_layout(
    xaxis_range=[0,5],
    legend_title_text = 'Hazardous??',
    updatemenus=[{
        'buttons': [
            {
                'label': 'Show All Asteroids',
                'method': 'update',
                'args': [{'visible': [True, True]}]
            },
            {
                'label': 'Hazardous Only',
                'method': 'update',
                'args': [{'visible': [False,True]}]
            },
            {
                "label": "Safe Only",
                "method": "update",
                "args": [{"visible": [True, False]}] # Shows safe, hides hazardous
            }
        ],
        "direction": "down",
        "showactive": True,
        "x": 0.0,
        'xanchor': 'left',
        "y": 1.05,
        'yanchor': 'bottom'
    }]
)
#Updating the x axis to only look at the truly near earth orbits

fig1.show()

#### Observations from Fig 1

There are a few outliers... Seing as they are not potentially hazardous and extremely far away from Earth's orbit I will exclude them by setting the x-axis limits to 0-5 AU.
\
Another cool observation is the hard curve that none of these astroids seem to ever go past for the hazardous asteroids. Upon looking it up it is a mathematical relationship between a and e that defines this line. `e = 1 - (1/a)` defines the Earth crossing boundary upon looking it up and this is why we se none of the hazardous ones go past it.

### Figure 2

For this visual I chose to display the data in a stacked bar chart. Similar to figure 1 I wanted to show the two groups we have of the hazardous and non-hazardous asteroids. We can see these broken down in the 4 asteroid size groups from left to right being the least danergous to the most dangerous. It instantly answers the question: "Are bigger space rocks more likely to be classified as potentially hazardous?" You can easily see what percentage of each size bar is filled with the "hazardous" color. It also highlights the sheer difference in volume between small and large asteroids. Because smaller asteroids are much more common (and easier to miss until recently), their overall bars will dwarf the giant "City Killer" bars, giving you an honest look at the population distribution.

In [18]:
threat_summary = df_clean.groupby(['size_short', 'pha']).size().reset_index(name='asteroid_count')

fig2 = px.bar(
    threat_summary,
    x='size_short',
    y='asteroid_count',
    color='pha',
    title='Threat Levels by Asteroid Size',
    labels={
        'size_short': 'Size Category',
        'asteroid_count': 'Num Asteroids',
        'pha': 'Potentially Hazardous'
    },
    category_orders={
        'size_short': [
            "Airburst/harmless",
            "Local Damage",
            "Regional Damage",
            "City Killer+"
        ]
    }
)

fig2.show()

#### Observations from Figure 2

I think it is interesting to note that nearly all of the small to medium asteroids are not potentially hazardous, further investigation into the data could help answer why this is!
\
I also find it interesting that the Regional Damage sized asteroids are the only group with a significant % of the asteroids being potentially hazardous. Again more data exploration would likely be required to answer why that is.

### Figure 3

This visualization uses an interactive Box Plot to compare the physical size distributions of asteroids across different orbital neighborhoods. It shows the "spread" of asteroid sizes. Each "box" visualizes the median size, the middle 50% of the data, and highlights extreme outliers (unusually giant or tiny asteroids) for each group. Now because asteroid sizes vary so drastically (some are the size of a house, others the size of a city), a standard graph scale makes it impossible to see both on the same screen. To solve this, the chart includes an interactive toggle:
* Log Scale
* Linear Scale

When you switch between them you can see why the log scale is so important for visualizing exponetial data like this!

#### Different classes and their meanings:
* Apollo (APO): These Earth-crossing asteroids have wide orbits that spend most of their time further from the Sun than Earth but dive inward to cross our orbital path.
* Amor (AMO): These Earth-approaching asteroids have orbits that stay entirely outside of Earth's path, meaning they get incredibly close to us but never actually cross our orbit.
* Aten (ATE): These Earth-crossing asteroids have narrow orbits that stay mostly closer to the Sun than Earth but stretch outward just enough to cross our orbital path.
* Interior to Earth (IEO): These rare asteroids have orbits contained entirely inside Earth's path, meaning they never cross our orbit and are exceptionally difficult to detect against the Sun's glare. (NOTE: This is likely a reason why the IEO sizes are bigger on avg because they need to be bigger to see them with our telescopes!)

In [19]:
fig3 = px.box(
    df_clean,
    x='class',
    y='diameter_km',
    color='class',
    title = 'Asteroid Diameter by Orbital Class',
    labels = {
        'class': 'Orbital Class',
        'diameter_km': 'Diameter (km)'
    },
    log_y=True #Need to turn this on because the diameters can scall but faster than a graph can show
)

fig3.update_layout(
    yaxis_title="Diameter (km, Log Scale)",
    showlegend=False, # It is redundant
    updatemenus=[{
        "buttons": [
            {
                "label": "Logarithmic Scale",
                "method": "relayout",
                "args": [{"yaxis.type": "log", "yaxis.title.text": "Diameter (km, Log Scale)"}]
            },
            {
                "label": "Linear Scale",
                "method": "relayout",
                "args": [{"yaxis.type": "linear", "yaxis.title.text": "Diameter (km, Linear Scale)"}]
            }
        ],
        "direction": "down",
        "showactive": True,
        "x": 0.0,
        'xanchor': 'left',
        "y": 1.05,
        'yanchor': 'bottom'
    }]
)

fig3.show()

### Figure 4

This visualization uses a Stacked Area Chart to display the running tally of asteroid discoveries over time, illustrating how our mapping of the solar system has accelerated It is similar to a line chart, but the space beneath the lines is filled in with color. The layers stack on top of each other, allowing you to see the growth of both the individual groups and the grand total simultaneously It clearly shows the exponential explosion in discoveries starting in the late 1990s and 2000s, while proving that the vast majority of our newly mapped space rocks are completely harmless. Now is that due to better technologies, more interest in near earth objects, both? Who knows but there is definitely more data to explore!

In [20]:
pivot_df = df_clean.groupby(['discovery_year', 'pha']).size().unstack(fill_value=0)

cumulative_df = pivot_df.cumsum()

timeline_df = cumulative_df.stack().reset_index(name='cumulative_count')

fig4 = px.area(
    timeline_df,
    x = 'discovery_year',
    y = 'cumulative_count',
    color = 'pha',
    title = ' Cumulative Asteroid Discoveries Over Years',
    labels={
        'discovery_year': 'Year of Discovery',
        'cumulative_count': 'Total Asteroids Discovered',
        'pha': 'Potentially Hazardous'
    }
)

fig4.show()

In [21]:
# Saving each chart to an interactive HTML file
fig1.write_html("chart1.html", include_plotlyjs="cdn")
fig2.write_html("chart2.html", include_plotlyjs="cdn")
fig3.write_html("chart3.html", include_plotlyjs="cdn")
fig4.write_html("chart4.html", include_plotlyjs="cdn")

The above code was used to generate the html files needed for the github page to load these visuals. The arguement include_plotlyjs='cdn' is to reduce the file size as I initially was running into size issues with the files at github uploads.

## Visualization Libraries

I chose **Plotly** for this project primarily because I use it regularly at work to develop Dash applications/tools within Databricks workspaces. Because of this, I already have a strong comfort level with its functions, interactive features, and syntax, which allowed me to quickly prototype these asteroid visualizations without a steep learning curve. Plotly also strikes a good balance between clean, high-level code and powerful interactivity for yourself and other users.

#### How to set up and import Plotly
If you are working outside of a pre-configured environment, you must manually install Plotly before you can use it.

* Open your terminal and run `pip install plotly`
* In your first cell (as seen in this notebook) run `import plotly.express as px`
* Then you are all set to go!
* Some easy charts to start with are `px.scatter`, `px.bar`, and `px.box`

Some helpful documentation can be found in the [Plotly Graphing Library](https://plotly.com/python/). This is fully open source and was created by Alex Johnson, Jack Parmer, Chris Parmer, and Matthew Sundquist in 2013.
\
Some limitations of Plotly ae its heavy files sizes since they are all interactive and with that being built into the backend it can take up a lot of space. There is also a performance ceiling so it was fine with the ~40k data points I had but if you get into the millions or billions then you may be in trouble. Writing out code for Plotly produces a really nice visual but can take a little longer than some of the other libraries we have used/learned about.

### Deploy instructions for dashboard
* Save the visuals produced by plotly as HTML files
* Add those files to a github repo
* Enable the Github Pages in the repo under Settings -> Pages
* Choose the branch and root then hit save and it will deploy the HTML Dashboard for you from the index.html file

Note: You do need some experience with html. I have developed a few apps with html so that practice helped me a lot in setting up this Github Page. Some helpful links for this can be found here:
* [Github Pages Quick Start](https://docs.github.com/en/pages/quickstart)
* [Getting Started with HTML](https://www.w3schools.com/html/)
* Generative AI can be a great tool as well but when using it you need to make sure you do not just blindly trust it. Also understanding all the code it writes and being able to understand the intricaces will help you grow as a developer too!